# Prompt Engineering Worksheet
## RAG and Prompt Engineering: From Concepts to Creation
### Module 2, Chapter 3 — Common Prompting Mistakes and How to Fix Them

This notebook gives you a hands-on way to work through the six prompt challenges from the course. Each challenge demonstrates one of the five common prompting mistakes — plus a bonus that puts all four elements together.


> 🎯 **Already done the setup? [Jump straight to the challenges!](#challenges)**

---

### Running this notebook

**If you're in Google Colab (recommended):**
You don't need to install anything. Just work through the cells from top to bottom, running each one with **Shift+Enter**.

New to Colab? Here's all you need to know:
- **Shift+Enter** runs a cell and moves to the next one
- **Ctrl+Enter** (or **Cmd+Enter** on Mac) runs a cell and stays on it
- Cells with `[ ]` on the left are code cells — they do something when you run them
- Cells like this one are text — they just display information
- If you see a warning that the notebook wasn't made by Google, click **Run Anyway** — that's normal for notebooks from outside Google

**If you're running locally (VS Code or Jupyter Lab):**
Make sure you have Python 3.8+ installed. The setup cell will install any required libraries automatically.

---

### Getting your API key

This notebook uses **Groq** by default — it's free, requires no credit card, and you can get a key in about two minutes:

1. Go to [console.groq.com](https://console.groq.com) (open it in a new tab so you don't lose your place here)
2. Click **API Keys** in the left sidebar → **Create API Key**
3. Copy the key — it starts with `gsk_...`
4. **Name your key `GROQ_API_KEY`** — this is what the setup cell looks for automatically

---

### Storing your key safely

**⚠️ Never paste your API key directly into a notebook you might share.** Instead, use one of these methods:

**In Google Colab (recommended):**
1. Click the 🔑 **Secrets** icon in the left sidebar
2. Click **Add new secret**
3. Name: `GROQ_API_KEY` — must be exact
4. Value: paste your key
5. Toggle **Notebook access** to ON
6. The setup cell will find it automatically

**Running locally:**
Set an environment variable before launching Jupyter:
```bash
export GROQ_API_KEY=your_key_here   # Mac/Linux
set GROQ_API_KEY=your_key_here      # Windows
```

**If all else fails:**
Paste your key directly into the `API_KEY` variable in the setup cell — just never share or publish the notebook with your key still in it.

---

### API options

If you prefer OpenAI or Anthropic, change the `PROVIDER` variable in the setup cell to `"openai"` or `"anthropic"`.
Free tiers and pricing can change — always check current details at the provider's website.

| Provider | Where to get a key | Free tier? |
|---|---|---|
| **Groq** (default) | [console.groq.com](https://console.groq.com) | Yes — no credit card needed |
| OpenAI | [platform.openai.com](https://platform.openai.com) | Pay-as-you-go — worksheet costs < $0.02 |
| Anthropic | [console.anthropic.com](https://console.anthropic.com) | Pay-as-you-go — worksheet costs < $0.02 |


## Setup

Run this cell first. It installs the required libraries, configures your API provider, and sets up the helper functions used throughout the notebook.

**Instructions:**
1. Choose your provider in the `PROVIDER` variable (`"groq"`, `"openai"`, or `"anthropic"`)
2. Paste your API key in the `API_KEY` variable — or store it in Colab Secrets (recommended)
3. Run the cell with Shift+Enter

**Using Colab Secrets?** Name your secret exactly `GROQ_API_KEY` (or `OPENAI_API_KEY` / `ANTHROPIC_API_KEY` if using those providers). The setup cell will find it automatically.

**If you see a warning about a deprecated package or a `FutureWarning` after running this cell,** the underlying library has probably released a new version since this notebook was published. The fix is usually `!pip install --upgrade groq` (or `openai` / `anthropic` depending on your provider) — run that in a new cell, then restart the runtime and run setup again.


In [ ]:
# ============================================================
# CONFIGURATION — edit these if needed, then run both cells
# ============================================================

PROVIDER = "groq"    # Options: "groq", "openai", "anthropic"
API_KEY  = None      # Leave as None to use Colab Secrets or env vars


In [ ]:
# @title ⚙️ Run setup { display-mode: "form" }

# Fallback defaults — these are overridden if you ran the config cell above
if "PROVIDER" not in dir():
    PROVIDER = "groq"
if "API_KEY" not in dir():
    API_KEY = None

import subprocess, sys, os

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

SECRET_NAMES = {
    "groq":      "GROQ_API_KEY",
    "openai":    "OPENAI_API_KEY",
    "anthropic": "ANTHROPIC_API_KEY",
}

# Resolve API key: manual entry → environment variable → Colab Secrets
secret_name = SECRET_NAMES.get(PROVIDER)
if API_KEY:
    API_KEY = API_KEY.strip()
    print(f"🔑 API key set manually")
else:
    API_KEY = os.environ.get(secret_name, "").strip() or None
    if API_KEY:
        print(f"🔑 API key loaded from environment variable ({secret_name})")
    else:
        try:
            from google.colab import userdata
            API_KEY = userdata.get(secret_name).strip()
            print(f"🔑 API key loaded from Colab Secrets ({secret_name})")
        except Exception:
            print(f"⚠️  No API key found.")
            print(f"   Please either:")
            print(f"   1. Set API_KEY = 'your_key_here' in the config cell above and re-run, or")
            print(f"   2. Add a Colab Secret named '{secret_name}' (see notebook intro), or")
            print(f"   3. Set an environment variable named '{secret_name}'.")

if not API_KEY:
    raise ValueError("No API key found. See instructions above.")

# Configure the chosen provider
if PROVIDER == "groq":
    install("groq")
    from groq import Groq
    groq_client = Groq(api_key=API_KEY)
    MODEL = "llama-3.3-70b-versatile"

elif PROVIDER == "openai":
    install("openai")
    from openai import OpenAI
    openai_client = OpenAI(api_key=API_KEY)
    MODEL = "gpt-4o-mini"

elif PROVIDER == "anthropic":
    install("anthropic")
    import anthropic
    anthropic_client = anthropic.Anthropic(api_key=API_KEY)
    MODEL = "claude-haiku-4-5-20251001"

else:
    raise ValueError(f"Unknown provider: '{PROVIDER}'. Choose 'groq', 'openai', or 'anthropic'.")


def ask(prompt, label="Response"):
    """Send a prompt and print the response with a label."""
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    print(f"PROMPT: {prompt}")
    print(f"{'-'*60}")

    if PROVIDER == "groq":
        response = groq_client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}]
        )
        print(response.choices[0].message.content)

    elif PROVIDER == "openai":
        response = openai_client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}]
        )
        print(response.choices[0].message.content)

    elif PROVIDER == "anthropic":
        response = anthropic_client.messages.create(
            model=MODEL,
            max_tokens=1024,
            messages=[{"role": "user", "content": prompt}]
        )
        print(response.content[0].text)


def compare(weak_prompt, strong_prompt):
    """Run both a weak and strong prompt and display them side by side."""
    ask(weak_prompt,   label="WEAK PROMPT OUTPUT")
    ask(strong_prompt, label="STRONG PROMPT OUTPUT")
    print(f"\n{'='*60}")
    print("  What changed?")
    print(f"{'='*60}")
    print(f"Weak:   {weak_prompt}")
    print(f"Strong: {strong_prompt}")


print(f"\n✅ Setup complete. Using {PROVIDER.upper()} with model: {MODEL}")
print("   Run any challenge cell below with Shift+Enter.")


<a name="challenges"></a>
---
## Challenge 1: Too Vague

**The mistake:** You want to know whether clowns are a good choice for your kid's birthday party. You type the first thing that comes to mind.

**What to watch for:** The weak prompt will produce something — probably a general overview of clown history, types of clowns, or the cultural significance of clowns. Not what you wanted.

**The fix:** Add the purpose, the audience, the format, and the length.

In [ ]:
# Challenge 1 — run this to see the weak vs strong output

weak   = "Tell me about clowns."

strong = """Write a 150-word overview of what to expect when hiring a clown 
for a five-year-old's birthday party. Friendly tone, practical details, 
no horror movie references."""

compare(weak, strong)

In [ ]:
# ✏️ YOUR TURN — write your own improved version of the clowns prompt
# Try a different audience, format, or angle and see what changes

my_prompt = """YOUR PROMPT HERE"""

ask(my_prompt, label="MY VERSION")

---
## Challenge 2: Too Much At Once

**The mistake:** You need to make the American Revolution interesting for your goth teenager who thinks history is boring. You also want it in French. And you want discussion questions. And a dramatic title. All in one prompt.

**What to watch for:** The weak prompt will produce something mediocre on all five tasks. Nothing will be done well.

**The fix:** Pick the one output you need most right now. Get that right. Then move on.

In [ ]:
# Challenge 2 — run this to see the weak vs strong output

weak   = """Summarize the American Revolution, translate it to French, 
rewrite it for a goth teenager who hates history, add three discussion 
questions, and suggest a dramatic title."""

strong = """Rewrite this summary of the American Revolution for a goth teenager 
who thinks history is boring. Dark tone, dramatic language, under 200 words.

Summary: The American Revolution (1765-1783) was a political upheaval in which 
thirteen American colonies broke free from British rule, establishing the United 
States of America. Key causes included taxation without representation, 
Enlightenment ideas about liberty, and colonial resentment of British control."""

compare(weak, strong)

In [ ]:
# ✏️ YOUR TURN — now try getting the French translation as a separate prompt
# Notice how much better the output is when you focus on one task

my_prompt = """YOUR PROMPT HERE — try asking for just the French translation, 
or just the discussion questions, as a separate focused request"""

ask(my_prompt, label="MY VERSION")

---
## Challenge 3: No Format Specified

**The mistake:** You want to cook dinner tonight with what's in your fridge. You ask for a recipe.

**What to watch for:** The weak prompt will produce a recipe for something — but it won't know what ingredients you have, how many people you're feeding, how much time you have, or what format you want. The model guesses on every axis.

**The fix:** Tell it what's in your fridge, your constraints, and the format you want.

In [ ]:
# Challenge 3 — run this to see the weak vs strong output

weak   = "Give me a recipe."

strong = """I have chicken thighs, garlic, canned tomatoes, and pasta. 
Give me a simple weeknight recipe for two people that takes under 30 minutes. 
Format as numbered steps with ingredients listed at the top."""

compare(weak, strong)

In [ ]:
# ✏️ YOUR TURN — use your actual fridge contents
# Try different format requests: shopping list, paragraph, table of options

my_ingredients = """LIST YOUR ACTUAL INGREDIENTS HERE"""

my_prompt = f"""I have {my_ingredients}. 
Give me a simple weeknight recipe for [NUMBER] people that takes under [TIME] minutes.
Format as [YOUR CHOSEN FORMAT]."""

ask(my_prompt, label="MY VERSION")

---
## Challenge 4: Forgetting the Audience

**The mistake:** You ask how GLP-1 works. The model doesn't know if you're a patient, a gym regular, or a doctor. It picks a general adult explanation that's probably right for nobody specific.

**What to watch for:** Run all three audience versions and compare them. Same question. Three completely different responses.

**The fix:** Tell the model who is reading the response.

In [ ]:
# Challenge 4 — weak prompt first

weak = "Explain how GLP-1 works."
ask(weak, label="WEAK PROMPT — no audience specified")

In [ ]:
# Challenge 4 — now run all three audience versions and compare

patient = """Explain how GLP-1 medications work in plain, friendly language 
for a patient who was just diagnosed with Type 2 diabetes and has never heard 
of this drug class before. Focus on what it means for their daily life. 
Keep it under 150 words."""

fitness = """Explain how GLP-1 medications work for a fitness enthusiast 
who is curious about Ozempic for weight loss. Focus on appetite suppression 
and the weight loss mechanism. Practical, direct tone. Under 150 words."""

doctor  = """Explain the mechanism of action of GLP-1 receptor agonists 
for an endocrinologist reviewing patient education materials. Include receptor 
binding, downstream signaling, and clinical implications. Under 200 words."""

ask(patient, label="VERSION A — Newly diagnosed Type 2 patient")
ask(fitness, label="VERSION B — Fitness enthusiast")
ask(doctor,  label="VERSION C — Endocrinologist")

In [ ]:
# ✏️ YOUR TURN — pick any topic and write three versions for three different audiences
# Good options: how the internet works, what inflation is, how vaccines work

topic = "YOUR TOPIC HERE"

audience_1 = f"Explain {topic} to a curious 10-year-old. Simple words, one analogy, under 100 words."
audience_2 = f"Explain {topic} to a smart non-specialist adult. No jargon, under 150 words."
audience_3 = f"Explain {topic} to a domain expert. Use precise terminology, under 200 words."

ask(audience_1, label="AUDIENCE 1 — 10-year-old")
ask(audience_2, label="AUDIENCE 2 — Non-specialist adult")
ask(audience_3, label="AUDIENCE 3 — Domain expert")

---
## Challenge 5: Not Iterating

**The mistake:** You ask about hurling. You want to understand the Irish sport before a match you're attending. The model confidently explains how to manage nausea.

**What to watch for:** The first response is wrong but confident. The first follow-up fixes the topic. The second follow-up makes it actually useful. Two short prompts beats one complicated one.

**The fix:** Treat the first response as a draft. One targeted follow-up at a time.

In [ ]:
# Challenge 5 — Step 1: the original weak prompt

weak = "Tell me about hurling."
ask(weak, label="STEP 1 — Weak prompt (prepare for nausea advice)")

In [ ]:
# Challenge 5 — Step 2: first follow-up to fix the topic

follow_up_1 = """Hurling is a traditional Irish sport played with a stick called 
a hurley and a small ball called a sliotar. Explain the basic rules of hurling."""

ask(follow_up_1, label="STEP 2 — First follow-up (correct the topic)")

In [ ]:
# Challenge 5 — Step 3: second follow-up to make it actually useful

follow_up_2 = """Summarize the essential rules of hurling in plain English 
for someone attending their very first match in an hour. 
Focus on what they need to know to follow the action. 
Use bullet points, keep it under 150 words."""

ask(follow_up_2, label="STEP 3 — Second follow-up (make it useful)")

In [ ]:
# ✏️ YOUR TURN — practice iterating on something you actually want to know
# Start with a vague prompt, then improve it in two follow-ups

# Step 1: start vague
my_step_1 = "YOUR VAGUE STARTING PROMPT HERE"
ask(my_step_1, label="MY STEP 1 — Starting prompt")

In [ ]:
# Step 2: first improvement
my_step_2 = "YOUR FIRST FOLLOW-UP HERE"
ask(my_step_2, label="MY STEP 2 — First improvement")

In [ ]:
# Step 3: second improvement
my_step_3 = "YOUR SECOND FOLLOW-UP HERE"
ask(my_step_3, label="MY STEP 3 — Second improvement")

---
## Challenge 6: Bonus — Put It All Together

**The task:** Write a compelling piece about climate change for your company newsletter.

**The exercise:** Build a strong prompt from scratch using all four elements. Fill in each element below, then let the cell assemble your final prompt and run it.

**The four elements:**
- **Task** — what specifically do you want the model to write?
- **Context** — who is the audience? what's the publication? what's the angle?
- **Format** — how long? what structure? what style?
- **Constraints** — what to avoid? what tone? any exclusions?

In [ ]:
# Challenge 6 — first, see what the weak prompt produces

weak = "Write something about climate change."
ask(weak, label="WEAK PROMPT — no elements specified")

In [ ]:
# ✏️ YOUR TURN — fill in each element, then run the cell

# Fill these in:
task        = "Write a [length]-word [content type] about climate change"  
# e.g. "Write a 300-word newsletter article intro about climate change"

context     = "The audience is [describe audience]. The publication is [describe publication]."
# e.g. "The audience is employees at a mid-size tech company. The publication is our internal newsletter."

format      = "Format it as [describe format]."
# e.g. "Format it as three short paragraphs with a compelling opening line."

constraints = "Keep the tone [describe tone]. Do not include [anything to exclude]."
# e.g. "Keep the tone optimistic and action-oriented. Do not include doom-and-gloom statistics."


# This assembles and runs your full prompt — no need to edit below
my_full_prompt = f"{task}\n\n{context}\n\n{format}\n\n{constraints}"

print("YOUR ASSEMBLED PROMPT:")
print("-" * 60)
print(my_full_prompt)
print("-" * 60)

ask(my_full_prompt, label="STRONG PROMPT — all four elements")

---
## The Fix Framework

When a response misses, run it through these three questions before re-prompting:

1. **What did I not specify?** — task, audience, angle, length
2. **What context did I assume the model had?** — background, role, situation
3. **What format did I expect but not request?** — structure, length, style

Nine times out of ten, one of those three questions points directly at the fix.

---

## Quick Reference: The Five Mistakes

| Mistake | Symptom | Fix |
|---|---|---|
| Too vague | Generic, unfocused output | Add audience, angle, length |
| Too much at once | Everything done badly | One task at a time |
| No format | Wrong structure | Specify the output shape |
| Wrong audience | Right answer, wrong room | Say who's reading it |
| Not iterating | Accepting a mediocre first response | One targeted follow-up at a time |

---

*RAG and Prompt Engineering: From Concepts to Creation — APImasters*  
*Instructor: Kirsten Hunter*